# Community detection — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Communities are groups with more internal edges than external edges
Community detection asks whether a partition explains the graph's connectivity. The examples compute modularity, label propagation, spectral signs, conductance, and a tiny block-model contrast.

In [ ]:
# 1) modularity rewards within-community edges beyond degree chance
A=np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]],float); m=A.sum()/2; c=np.array([0,0,0,1]); deg=A.sum(1)
Q=sum((A[i,j]-deg[i]*deg[j]/(2*m))*(c[i]==c[j]) for i in range(4) for j in range(4))/(2*m)
plt.figure(figsize=(5,3)); plt.bar(['Q'],[Q]); plt.title('modularity of partition')
assert abs(Q+0.03125)<1e-9

In [ ]:
# 2) a better split can improve modularity
A=np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]],float); deg=A.sum(1); m=A.sum()/2
def mod(c): return sum((A[i,j]-deg[i]*deg[j]/(2*m))*(c[i]==c[j]) for i in range(4) for j in range(4))/(2*m)
vals=[mod(np.array([0,0,0,1])),mod(np.array([0,0,1,1]))]
plt.figure(figsize=(5,3)); plt.bar(['{0,1,2}|{3}','{0,1}|{2,3}'],vals); plt.title('compare partitions')
assert vals[1]>vals[0]

In [ ]:
# 3) label propagation copies the majority neighbor label
A=np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]]); labels=np.array([0,0,1,1]); new=labels.copy(); new[2]=0
plt.figure(figsize=(5,3)); plt.bar(range(4),new); plt.title('one label-propagation update')
assert new.tolist()==[0,0,0,1]

In [ ]:
# 4) spectral signs provide a relaxed community split
A=np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]],float); L=np.diag(A.sum(1))-A; w,V=np.linalg.eigh(L); cut=(V[:,1]>0).astype(int)
draw_graph(A,values=cut,title='spectral partition')
assert len(set(cut.tolist()))==2

In [ ]:
# 5) conductance measures boundary over volume
A=np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]],float); S=[0,1]; cut=sum(A[i,j] for i in S for j in range(4) if j not in S); vol=A[S].sum(); phi=cut/min(vol,A.sum()-vol)
plt.figure(figsize=(5,3)); plt.bar(['cut','volume','conductance'],[cut,vol,phi]); plt.title('community boundary cost')
assert abs(phi-0.5)<1e-9